# AMD Stock Linear Regression Analysis

This notebook performs exploratory data analysis and linear regression on AMD historical stock price data.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Data Loading

Load AMD stock data from CSV file. The file should be located in the same directory as this notebook or in a `data/` subdirectory.

In [ ]:
# Define file path (supports both local and data subdirectory)
file_paths = [
    'AMD.csv',
    'data/AMD.csv',
    Path.cwd() / 'AMD.csv'
]

df = None
for file_path in file_paths:
    try:
        df = pd.read_csv(file_path)
        print(f"✓ Data loaded successfully from: {file_path}")
        break
    except FileNotFoundError:
        continue

if df is None:
    raise FileNotFoundError(
        "AMD.csv not found. Please ensure the file is in the current directory or 'data/' subdirectory."
    )

# Display first few rows
df.head()

## Exploratory Data Analysis

In [ ]:
# Dataset info
print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nDataset Info:")
df.info()

In [ ]:
# Display first and last rows
print("First 5 rows:")
print(df.head())
print("\nLast 5 rows:")
print(df.tail())

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Column statistics
print("Mean of Low prices:", df["Low"].mean())
print("Median of Low prices:", df["Low"].median())
print("Std Dev of Low prices:", df["Low"].std())

## Data Cleaning

In [ ]:
# Check for duplicates
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print("Removing duplicates...")
    df.drop_duplicates(inplace=True)
    print(f"Dataset shape after removing duplicates: {df.shape}")

In [ ]:
# Check for missing values
print("Missing values per column:")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
# Verify data quality
print(f"Dataset shape after cleaning: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n✓ Data cleaning complete!")

## Data Visualization

In [ ]:
# Plot closing price over time
plt.figure(figsize=(14, 6))
plt.plot(pd.to_datetime(df['Date']), df['Close'], linewidth=1, alpha=0.8)
plt.title('AMD Stock Closing Price Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of closing prices
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(df['Close'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Closing Prices')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].hist(df['Volume'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution of Volume')
axes[0, 1].set_xlabel('Volume')
axes[0, 1].set_ylabel('Frequency')

axes[1, 0].scatter(df['High'], df['Low'], alpha=0.5, s=10)
axes[1, 0].set_title('High vs Low Prices')
axes[1, 0].set_xlabel('High Price ($)')
axes[1, 0].set_ylabel('Low Price ($)')

axes[1, 1].boxplot([df['Open'], df['Close'], df['High'], df['Low']], 
                     labels=['Open', 'Close', 'High', 'Low'])
axes[1, 1].set_title('Price Distribution Comparison')
axes[1, 1].set_ylabel('Price ($)')

plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
# Calculate correlation matrix
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - AMD Stock Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation Matrix:")
print(correlation_matrix)

## Linear Regression Model

Build a simple linear regression model to predict closing prices based on other features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Prepare features and target
X = df[['Open', 'High', 'Low', 'Volume']].copy()
y = df['Close'].copy()

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

In [ ]:
# Train linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print("✓ Model trained successfully!")

In [ ]:
# Evaluate model performance
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

print("Model Performance Metrics:")
print(f"\nTraining Set:")
print(f"  R² Score: {train_r2:.4f}")
print(f"  RMSE: ${train_rmse:.4f}")
print(f"  MAE: ${train_mae:.4f}")
print(f"\nTesting Set:")
print(f"  R² Score: {test_r2:.4f}")
print(f"  RMSE: ${test_rmse:.4f}")
print(f"  MAE: ${test_mae:.4f}")

In [ ]:
# Display model coefficients
print("Model Coefficients:")
print(f"Intercept: {model.intercept_:.4f}")
print("\nFeature Coefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature}: {coef:.6f}")

In [ ]:
# Visualize predictions vs actual values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_pred_train, alpha=0.5, s=20)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Training Set (R² = {train_r2:.4f})')
axes[0].grid(True, alpha=0.3)

# Testing set
axes[1].scatter(y_test, y_pred_test, alpha=0.5, s=20, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Price ($)')
axes[1].set_ylabel('Predicted Price ($)')
axes[1].set_title(f'Testing Set (R² = {test_r2:.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze residuals
residuals_train = y_train - y_pred_train
residuals_test = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals distribution
axes[0].hist(residuals_train, bins=50, alpha=0.5, label='Training', edgecolor='black')
axes[0].hist(residuals_test, bins=50, alpha=0.5, label='Testing', edgecolor='black')
axes[0].set_xlabel('Residuals ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Residuals')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals vs predicted values
axes[1].scatter(y_pred_test, residuals_test, alpha=0.5, s=20)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].set_title('Residuals vs Predicted Values (Test Set)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean of residuals (Training): {residuals_train.mean():.6f}")
print(f"Mean of residuals (Testing): {residuals_test.mean():.6f}")
print(f"Std Dev of residuals (Training): {residuals_train.std():.4f}")
print(f"Std Dev of residuals (Testing): {residuals_test.std():.4f}")

## Summary

This analysis provided:
- Comprehensive exploratory data analysis of AMD stock data
- Data cleaning and validation
- Correlation analysis between features
- Linear regression model to predict closing prices
- Model evaluation metrics and visualizations
- Residual analysis for model diagnostics